In [77]:
import numpy as np
import gym
from gym import spaces
import random
from gym.utils import seeding

class conn(gym.Env):
  """
  Custom Environment that follows gym interface.
  This is a simple env where the agent must learn to go always left.
  """
  # Because of google colab, we cannot implement the GUI ('human' render mode)
  metadata = {'render.modes': ['console']}
  # Define constants for clearer code


  def __init__(self,connectors_dicc, list_signals, connector, len_test_conn, max_categories, state_space=10000):
      super(conn, self).__init__()
      # numero de pins connector -1 due to PartNumber
      self.grid_size=len_test_conn
      self.connectors_dicc=connectors_dicc
      self.signals=list_signals # different signals to plug into a PIN Categories
      # Example when using discrete actions,
      self.state_space=state_space
      self.max_categories=max_categories
      self.action_space = spaces.Discrete(self.state_space)
      self.observation_space = spaces.Box(low=0, high=self.max_categories,
                                        shape=(self.grid_size,), dtype=np.float32)
      self.state = np.zeros(self.grid_size, dtype='int')
      self.agent_pos=1
      self.terminal= False
      self.lr=.5
      self.this_connector=connector
      self.reward = 0
      self._seed()
      self.cat_prefered_signals=[]
      for key in self.connectors_dicc[connector].keys():
        if key != 'PartNumber':
            self.cat_prefered_signals.append(self.connectors_dicc[connector][key]['Cat_PreferredSignal'])
        
        
  def _seed(self, seed=None):
       self.np_random, seed = seeding.np_random(seed)
       return [seed]

  def reset(self):
      """
      Important: the observation must be a numpy array
      :return: (np.array)
      :return:
      """
      self.state = np.zeros(self.grid_size, dtype='int')
      self.agent_pos=1
      self.reward=0

      return self.state

  def render(self, mode='console'):
      if mode != 'console':
          raise NotImplementedError()
      # agent is represented as a cross, rest as a dot

      print("x", end="")


  def close(self):
      pass


  def calculate_reward(self,combination):
    reward=0
    for i in range(0, len(combination)):
        signal=combination[i]
        if self.connectors_dicc[self.this_connector][str(self.agent_pos+1)]['Cat_Signal']==signal:
            reward = reward + 10
        elif signal in self.cat_prefered_signals:
            reward = reward + 1
        else:
            reward=reward -1
    return reward

  def step(self, action):
      self.terminal= self.agent_pos < self.grid_size + 1
      #print(action)
      if self.terminal:
        combination=[]
        for i in range(0, self.grid_size ):
            random.seed(self.agent_pos)
            ts=random.random()
            if ts > self.lr:
                signal=random.randint(0, self.max_categories)
                combination.append(int(signal))
            else:
                signal=self.connectors_dicc[self.this_connector][i+1]['Cat_Signal']
                combination.append(int(signal))
                
        print(combination)
        self.state = np.array(combination)

        # Null reward everywhere except when reaching the goal (left of the grid)
        self.reward = self.calculate_reward(combination)
        # Are we at the left of the grid?
        #self.terminal = bool(self.agent_pos >= 1000) or self.reward > self.grid_size*7
        self.terminal = self.reward > self.grid_size*8
        # Account for the boundaries of the grid
        self.agent_pos = self.agent_pos = self.agent_pos + 1


        # Optionally we can pass additional info, we are not using that for now
        info = {}

        return self.state, self.reward, self.terminal, info
      else:
          print(self.reward)
          print(self.state)

          info = {}

          return self.state, self.reward, self.terminal, info

In [2]:
from agent_pinner.pre_processing import *

In [3]:
data = pd.read_csv("agent_pinner/data/inline_pins.csv")
data, d, max_categories = preprocess_data(data)

In [4]:
connectors_dicc=create_connector_dicc(data)

In [5]:
len_test_conn=len(connectors_dicc['TO_111_LH_(121)'].keys()) - 1
list_signals=list(data['Cat_Signal Name'].unique())
state_space=(max_categories)

In [6]:
state_space

510

In [7]:
from stable_baselines3.common.env_checker import check_env

In [8]:
#connectors_dicc, list_signals, connector, len_test_conn, max_categories, state_space=10000

In [55]:
env = conn(connectors_dicc,list_signals ,'TO_111_LH_(121)',len_test_conn, max_categories,state_space)

In [38]:
check_env(env, warn=True)

[122, 112, 21, 22, 0, 85, 87, 101, 104, 105, 106, 109, 110, 168, 113, 13, 123, 141, 145, 154, 14, 170, 183, 185, 186, 187, 188, 192, 198, 200, 363, 364, 365, 371, 372, 411, 436, 437, 438, 439]
[485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485]
[122, 112, 21, 22, 0, 85, 87, 101, 104, 105, 106, 109, 110, 168, 113, 13, 123, 141, 145, 154, 14, 170, 183, 185, 186, 187, 188, 192, 198, 200, 363, 364, 365, 371, 372, 411, 436, 437, 438, 439]
[122, 112, 21, 22, 0, 85, 87, 101, 104, 105, 106, 109, 110, 168, 113, 13, 123, 141, 145, 154, 14, 170, 183, 185, 186, 187, 188, 192, 198, 200, 363, 364, 365, 371, 372, 411, 436, 437, 438, 439]
[379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379]
[420, 420, 420, 420

In [56]:
obs = env.reset()
env.render()

x

In [57]:
print(env.observation_space)
print(env.action_space)
print(env.action_space.sample())

Box(40,)
Discrete(510)
172


In [58]:
n_steps = 1000
GO_LEFT=random.randint(0, max_categories )
for step in range(n_steps):
    print("Step {}".format(step + 1))
    GO_LEFT=random.randint(0, max_categories )
    obs, reward, done, info = env.step(GO_LEFT)
    print('obs=', obs, 'reward=', reward, 'done=', done)
    env.render()
    if done:
        print("Goal reached!", "reward=", reward)
        break

Step 1
[122, 391, 21, 22, 241, 403, 87, 101, 199, 105, 1, 136, 117, 52, 15, 13, 4, 195, 496, 154, 14, 170, 183, 185, 186, 187, 188, 192, 198, 200, 284, 51, 365, 440, 372, 458, 364, 495, 438, 343]
obs= [122 391  21  22 241 403  87 101 199 105   1 136 117  52  15  13   4 195
 496 154  14 170 183 185 186 187 188 192 198 200 284  51 365 440 372 458
 364 495 438 343] reward= 13 done= False
xStep 2
[485, 46, 21, 376, 437, 85, 310, 101, 220, 411, 509, 190, 227, 461, 113, 13, 123, 141, 145, 269, 14, 170, 183, 185, 186, 184, 345, 509, 407, 200, 465, 404, 185, 492, 372, 204, 236, 127, 438, 256]
obs= [485  46  21 376 437  85 310 101 220 411 509 190 227 461 113  13 123 141
 145 269  14 170 183 185 186 184 345 509 407 200 465 404 185 492 372 204
 236 127 438 256] reward= -2 done= False
xStep 3
[122, 189, 242, 33, 465, 132, 98, 240, 281, 105, 77, 109, 110, 199, 343, 81, 302, 141, 421, 242, 471, 365, 218, 185, 227, 68, 49, 192, 198, 200, 398, 154, 365, 293, 372, 299, 436, 437, 14, 507]
obs= [122 189 

 436 437 447 439] reward= 1 done= False
xStep 84
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 85
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 86
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435

 436 437 447 439] reward= 1 done= False
xStep 139
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 140
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 141
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 

 436 437 447 439] reward= 1 done= False
xStep 193
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 194
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 195
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 

xStep 244
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 245
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 246
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374

...

 436 437 447 439] reward= 1 done= False
xStep 755
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 756
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 757
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 

 436 437 447 439] reward= 1 done= False
xStep 807
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 808
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 809
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 

 436 437 447 439] reward= 1 done= False
xStep 858
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 859
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 860
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 

 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 908
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 909
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 910
1
[122  16  21 326  65 141 225 270  66 105 106 

1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 959
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439] reward= 1 done= False
xStep 960
1
[122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 185 186 236  80 107 198  87 311   4 365  57 372 411
 436 437 447 439]
obs= [122  16  21 326  65 141 225 270  66 105 106 435  90 160 113  74 348 452
 208 431 374 116 158 1

In [65]:
from stable_baselines3.common.vec_env import DummyVecEnv

In [66]:
env_train = DummyVecEnv([lambda: conn(connectors_dicc,list_signals ,'TO_111_LH_(121)',len_test_conn, max_categories,state_space)])

In [67]:
from stable_baselines3 import DQN

In [68]:
model = DQN('MlpPolicy', env_train, verbose=2)

Using cpu device


In [76]:
model.learn(10000)

[122, 112, 21, 22, 0, 85, 87, 101, 104, 105, 106, 109, 110, 168, 113, 13, 123, 141, 145, 154, 14, 170, 183, 185, 186, 187, 188, 192, 198, 200, 363, 364, 365, 371, 372, 411, 436, 437, 438, 439]
[485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485, 485]
[122, 112, 21, 22, 0, 85, 87, 101, 104, 105, 106, 109, 110, 168, 113, 13, 123, 141, 145, 154, 14, 170, 183, 185, 186, 187, 188, 192, 198, 200, 363, 364, 365, 371, 372, 411, 436, 437, 438, 439]
[122, 112, 21, 22, 0, 85, 87, 101, 104, 105, 106, 109, 110, 168, 113, 13, 123, 141, 145, 154, 14, 170, 183, 185, 186, 187, 188, 192, 198, 200, 363, 364, 365, 371, 372, 411, 436, 437, 438, 439]
[379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379, 379]
[420, 420, 420, 420

49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439

 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 41

49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439

49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439

...

 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 41

[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
4

 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 41

[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
4

49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439]
49
[122 112  21  22   0  85  87 101 104 105 106 109 110 168 113  13 123 141
 145 154  14 170 183 185 186 187 188 192 198 200 363 364 365 371 372 411
 436 437 438 439

In [74]:
obser=[121,
 112,
 21,
 22,
 0,
 85,
 87,
 101,
 104,
 105,
 106,
 109,
 110,
 168,
 113,
 13,
 123,
 141,
 145,
 154,
 14,
 170,
 183,
 185,
 186,
 187,
 188,
 192,
 198,
 200,
 363,
 364,
 365,
 371,
 372,
 411,
 436,
 437,
 438,
 439]

In [75]:
model.predict(obser)

(268, None)